# POS tags + CRF

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
import os
import sys
import codecs
import os.path
from sklearn.model_selection import train_test_split
from gensim.models import FastText, KeyedVectors
from nltk.tag.crf import CRFTagger
import pycrfsuite
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings("ignore")
from gensim.models import Word2Vec
from scipy.sparse import hstack, csr_matrix

sys.path.append(os.path.abspath("../.."))

from preprocessing.utils import *

In [3]:
def load_pres(fname):
    alltxts, alllabs = [], []
    s = codecs.open(fname, 'r', 'utf-8')
    while True:
        txt = s.readline()
        if len(txt) < 5:
            break
        lab = re.sub(r"<[0-9]*:[0-9]*:(.)>.*", "\\1", txt)
        txt = re.sub(r"<[0-9]*:[0-9]*:.>(.*)", "\\1", txt).strip()
        alllabs.append("M" if "M" in lab else "C")
        alltxts.append(txt)
    return alltxts, alllabs

In [4]:
fname = "../../../data/corpus.tache1.learn.utf8"
alltxts, alllabs = load_pres(fname)

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    alltxts, alllabs,
    test_size=0.2,
    random_state=42,
    stratify=alllabs
)
print(f"Train: {len(X_train)} | Val: {len(X_val)}")

Train: 45930 | Val: 11483


### POS tagging with spaCy (French model)

In [6]:
import spacy

print("Loading spaCy French model...")
nlp = spacy.load("fr_core_news_sm")

def pos_tag_sentences(texts):
    """
    Returns list of docs, each doc is a list of (word, POS) tuples.
    Uses spaCy pipeline in batch mode for speed.
    """
    tagged = []
    for doc in nlp.pipe(texts, batch_size=512, disable=["ner", "parser"]):
        tagged.append([(token.text.lower(), token.pos_) for token in doc if not token.is_space])
    return tagged

print("POS tagging train set...")
train_tagged = pos_tag_sentences(X_train)   # [(word, POS), ...]
print("POS tagging val set...")
val_tagged   = pos_tag_sentences(X_val)
print("POS tagging done.")

Loading spaCy French model...
POS tagging train set (this may take a few minutes)...
POS tagging val set...
POS tagging done.


### Build CRF training format

Input:  [(word, POS), ...]  per sentence

Output: ["C", "C", ..., "M", "M", ...]  (same label for all words in sentence)

In [7]:
def make_crf_data(tagged_sentences, labels):
    """
    tagged_sentences : list of [(word, pos), ...]
    labels           : list of "C" or "M" per sentence
    Returns X (list of feature dicts per sentence) and y (list of label lists)
    """
    X, y = [], []
    for tagged, lab in zip(tagged_sentences, labels):
        if not tagged:
            continue
        X.append(sent2features(tagged))
        y.append([lab] * len(tagged))   # every word gets the sentence label
    return X, y

### Feature extraction

Features per token: POS, surrounding POS, morphological cues

In [8]:
def word2features(sent, i):
    """
    sent : list of (word, POS) tuples
    i    : index of current token
    """
    word, pos = sent[i]

    features = {
        'bias':           1.0,
        'word':           word,
        'pos':            pos,
        'is_first':       i == 0,
        'is_last':        i == len(sent) - 1,
        'is_capitalized': word[0].isupper() if word else False,
        'is_all_lower':   word.islower(),
        'is_numeric':     word.isdigit(),
        'has_hyphen':     '-' in word,
        'prefix-2':       word[:2],
        'prefix-3':       word[:3],
        'suffix-2':       word[-2:],
        'suffix-3':       word[-3:],
    }

    # Previous token features
    if i > 0:
        prev_word, prev_pos = sent[i - 1]
        features.update({
            'prev_word':   prev_word,
            'prev_pos':    prev_pos,
            'prev_suffix': prev_word[-3:],
        })
    else:
        features['BOS'] = True   # Beginning of Sentence

    # Next token features
    if i < len(sent) - 1:
        next_word, next_pos = sent[i + 1]
        features.update({
            'next_word':   next_word,
            'next_pos':    next_pos,
            'next_suffix': next_word[-3:],
        })
    else:
        features['EOS'] = True   # End of Sentence

    # POS bigram context
    if i > 0:
        features['prev_pos+pos'] = sent[i-1][1] + '+' + pos
    if i < len(sent) - 1:
        features['pos+next_pos'] = pos + '+' + sent[i+1][1]

    return features

In [9]:
def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


### Build datasets

In [10]:
print("Building CRF datasets...")
X_train_crf, y_train_crf = make_crf_data(train_tagged, y_train)
X_val_crf,   y_val_crf   = make_crf_data(val_tagged,   y_val)


Building CRF datasets...


### Train CRF

In [11]:
import sklearn_crfsuite


print("\nTraining CRF...")
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,            # L1 regularization
    c2=0.1,            # L2 regularization
    max_iterations=200,
    all_possible_transitions=True
)
crf.fit(X_train_crf, y_train_crf)
print("CRF training done.")



Training CRF...
CRF training done.


### Word-level evaluation

In [12]:
y_val_flat   = [lab for sent in y_val_crf   for lab in sent]
y_pred_flat  = [lab for sent in crf.predict(X_val_crf) for lab in sent]

print("\n--- Word-level evaluation ---")
print(classification_report(y_val_flat, y_pred_flat, target_names=["C", "M"]))



--- Word-level evaluation ---
              precision    recall  f1-score   support

           C       0.93      0.95      0.94    248914
           M       0.68      0.62      0.65     47485

    accuracy                           0.89    296399
   macro avg       0.81      0.78      0.79    296399
weighted avg       0.89      0.89      0.89    296399

